# Table of Content
01. Import Libraries
02. Import Data
03. Customer Profiling Conditions
04. First Approach - with overlap issue
05. Second Approach - successful

# 01. Import Libraries

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import os

In [2]:
# Adjust the setting to view all columns in this notebook
pd.options.display.max_columns = None

# 02. Import Data

In [3]:
# Define the main project folder path
path = r'C:\Users\saich\Desktop\CareerFoundry\Data Immersion\Achievement 4 Python Fundamentals for Data Analysts\04-2023 Instacart Basket Analysis (github)'

In [4]:
# Import 'orders_products_all' data set from 'Prepared Data' folder
ords_prods_all = pd.read_pickle(os.path.join(path, '02 Data', 'Prepared Data', 'orders_products_all.pkl'))

In [5]:
ords_prods_all.head()

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,prices,price_range,busiest_day,busiest_days,busiest_period_of_day,max_order,loyalty_flag,avg_price,spender_flag,median_order_interval,order_frequency_flag,gender,state,age,date_joined,dependant_counts,family_status,income,_merge
0,2539329,1,1,2,8,NaN,196,1,0,Soda,77,7,9.0,Mid-range product,Regular busy,Regular busy,Average orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer,Female,Alabama,31,2/17/2019,3,married,40423,both
1,2398795,1,2,3,7,15.0,196,1,1,Soda,77,7,9.0,Mid-range product,Regular busy,Least busy,Average orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer,Female,Alabama,31,2/17/2019,3,married,40423,both
2,473747,1,3,3,12,21.0,196,1,1,Soda,77,7,9.0,Mid-range product,Regular busy,Least busy,Most orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer,Female,Alabama,31,2/17/2019,3,married,40423,both
3,2254736,1,4,4,7,29.0,196,1,1,Soda,77,7,9.0,Mid-range product,Least busy,Least busy,Average orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer,Female,Alabama,31,2/17/2019,3,married,40423,both
4,431534,1,5,4,15,28.0,196,1,1,Soda,77,7,9.0,Mid-range product,Least busy,Least busy,Most orders,10,New customer,6.367797,Low spender,20.5,Non-frequent customer,Female,Alabama,31,2/17/2019,3,married,40423,both


In [6]:
ords_prods_all.shape

(32399732, 31)

# 03. Customer Profiling Conditions

Customers are grouped into different profiles based on the conditions below: 

- if 'age' >= 65 --> <b>Senior citizen</b><br><br>

- if 'age' < 65, 'family_status' = single or divorced/widowed, 'dependant_counts' = 0 --> <b>Single adult</b><br><br>

- if 'age' < 65, 'family_status' = married, 'dependant_counts' <= 2 --> <b>Small family</b>
- if 'age' < 65, 'family_status' = single or divorced/widowed, 'dependant_counts' = 1 or 2 --> <b>Small family</b><br><br>

- if 'age' < 65, 'family_status' = married, single or divorced/widowed, 'dependant_counts’ > 2 --> <b>Big family</b>
- if 'age' < 65, 'family_status' = single or divorced/widowed, 'dependant_counts' > 2 --> <b>Big family</b>
- if 'age' < 65, 'family_status' = living with parents and siblings --> <b>Big family</b><br><br>

- if 'age' < 65, 'dependant_counts' > 0, and 'department_id' = 18 --> <b>Young parent</b>

In [7]:
# Number of unique users
ords_prods_all['user_id'].nunique()

206209

# 04. First Approach - with overlap issue

In [8]:
# Define the 'Senior citizen' profile
ords_prods_all.loc[(ords_prods_all['age'] >= 65), 'customer_profile'] = 'Senior citizen'

In [9]:
# Define the 'Single adult' profile
ords_prods_all.loc[(ords_prods_all['age'] < 65) & 
                   (ords_prods_all['family_status'].isin(['single', 'divorced/widowed'])) & 
                   (ords_prods_all['dependant_counts'] == 0), 
                   'customer_profile'] = 'Single adult'

In [10]:
# Define the 'Small family' profile
ords_prods_all.loc[(ords_prods_all['age'] < 65) & 
                   (ords_prods_all['family_status'] == 'married') & 
                   (ords_prods_all['dependant_counts'] <= 2), 
                   'customer_profile'] = 'Small family'

In [11]:
ords_prods_all.loc[(ords_prods_all['age'] < 65) & 
                   (ords_prods_all['family_status'].isin(['single', 'divorced/widowed'])) & 
                   (ords_prods_all['dependant_counts'].isin([1, 2])), 
                   'customer_profile'] = 'Small family'

In [12]:
# Define the 'Big family' profile
ords_prods_all.loc[(ords_prods_all['age'] < 65) & 
                   (ords_prods_all['family_status'].isin(['married', 'single', 'divorced/widowed'])) & 
                   (ords_prods_all['dependant_counts'] > 2), 
                   'customer_profile'] = 'Big family'

In [13]:
ords_prods_all.loc[(ords_prods_all['age'] < 65) & 
                   (ords_prods_all['family_status'] == 'living with parents and siblings'), 
                   'customer_profile'] = 'Big family'

In [14]:
# Define the 'Young parent' profile
ords_prods_all.loc[(ords_prods_all['age'] < 65) & 
                   (ords_prods_all['dependant_counts'] > 0) & 
                   (ords_prods_all['department_id'] == 18), 
                   'customer_profile'] = 'Young parent'

In [15]:
# Check the output
ords_prods_all['customer_profile'].value_counts(dropna = False)

customer_profile
Small family      10680872
Senior citizen     8573751
Big family         6934007
Single adult       5976558
Young parent        234544
Name: count, dtype: int64

No missing values in 'customer_profile' column.

In [16]:
# Check the number of distinct users in each customer profile. Total should be 206,209 distinct users. 
ords_prods_all.groupby(['customer_profile'])['user_id'].nunique()

customer_profile
Big family        44430
Senior citizen    54729
Single adult      37944
Small family      69106
Young parent      18565
Name: user_id, dtype: int64

The total distinct users from each profile is 224,774, which is more than 206,209. <br>
This means some users has more than one customer profiles.

# 05. Second Approach - successful

#### (i) Define 'Senior citizen', 'Single adult', 'Small family' and 'Big family' profiles first which do not involve 'department_id' variable

In [17]:
# Define the 'Senior citizen' profile
ords_prods_all.loc[(ords_prods_all['age'] >= 65), 'customer_profile_2'] = 'Senior citizen'

In [18]:
# Define the 'Single adult' profile
ords_prods_all.loc[(ords_prods_all['age'] < 65) & 
                   (ords_prods_all['family_status'].isin(['single', 'divorced/widowed'])) & 
                   (ords_prods_all['dependant_counts'] == 0), 
                   'customer_profile_2'] = 'Single adult'

In [19]:
# Define the 'Small family' profile
ords_prods_all.loc[(ords_prods_all['age'] < 65) & 
                   (ords_prods_all['family_status'] == 'married') & 
                   (ords_prods_all['dependant_counts'] <= 2), 
                   'customer_profile_2'] = 'Small family'

In [20]:
ords_prods_all.loc[(ords_prods_all['age'] < 65) & 
                   (ords_prods_all['family_status'].isin(['single', 'divorced/widowed'])) & 
                   (ords_prods_all['dependant_counts'].isin([1, 2])), 
                   'customer_profile_2'] = 'Small family'

In [21]:
# Define the 'Big family' profile
ords_prods_all.loc[(ords_prods_all['age'] < 65) & 
                   (ords_prods_all['family_status'].isin(['married', 'single', 'divorced/widowed'])) & 
                   (ords_prods_all['dependant_counts'] > 2), 
                   'customer_profile_2'] = 'Big family'

In [22]:
ords_prods_all.loc[(ords_prods_all['age'] < 65) & 
                   (ords_prods_all['family_status'] == 'living with parents and siblings'), 
                   'customer_profile_2'] = 'Big family'

In [23]:
# Check the output
ords_prods_all['customer_profile_2'].value_counts(dropna = False)

customer_profile_2
Small family      10821505
Senior citizen     8573751
Big family         7027918
Single adult       5976558
Name: count, dtype: int64

No missing values in 'customer_profile_2' column.

In [24]:
# Check the number of distinct users in each customer profile. Total should be 206,209 distinct users. 
ords_prods_all.groupby(['customer_profile_2'])['user_id'].nunique()

customer_profile_2
Big family        44430
Senior citizen    54729
Single adult      37944
Small family      69106
Name: user_id, dtype: int64

At this stage, the total distinct users from each profile is 206,209. This means each user has only one customer profile at this stage.

#### (ii) Define 'Young parent' profile which involves 'department_id' variable

Since one user ID could have different values of 'department_id' in different records, the 'Young parent' profile is defined differently from the four profiles above. 

In [25]:
# 1. Define the condition of 'Young parent' profile
condition = ((ords_prods_all['age'] < 65) & 
            (ords_prods_all['dependant_counts'] > 0) & 
            (ords_prods_all['department_id'] == 18))  # department_id '18' is babies department

In [26]:
# 2. Find the list of unique 'user_id' that meet the condition
young_parent_user_id = ords_prods_all.loc[condition, 'user_id'].unique()

In [27]:
young_parent_user_id

array([  290,   420,  1613, ...,  2625, 90276, 21688])

In [28]:
len(young_parent_user_id)

18565

In [29]:
# 3. Change the 'customer_profile' of 'young_parent_user_id' to 'Young parent'
ords_prods_all.loc[ords_prods_all['user_id'].isin(young_parent_user_id), 'customer_profile_2'] = 'Young parent'

In [30]:
# Check the output
ords_prods_all['customer_profile_2'].value_counts(dropna = False)

customer_profile_2
Senior citizen    8573751
Small family      7497037
Single adult      5976558
Young parent      5513951
Big family        4838435
Name: count, dtype: int64

No missing values in 'customer_profile_2' column.

In [31]:
# Check the number of distinct users in each customer profile. Total should be 206,209 distinct users. 
ords_prods_all.groupby(['customer_profile_2'])['user_id'].nunique()

customer_profile_2
Big family        37181
Senior citizen    54729
Single adult      37944
Small family      57790
Young parent      18565
Name: user_id, dtype: int64

The total number of distinct users from each profile is 206,209. This means there is no user having more than one customer profile. 